In [17]:
import duckdb
import os

In [ ]:
con = duckdb.connect("warehouse.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS raw;")

In [66]:
def ingestion_query(source_path,table_name="raw.yellow_tripdata",first_load=False):

    main_command=f"CREATE OR REPLACE TABLE  {table_name} AS " if first_load else f"INSERT INTO {table_name} "

    query=f"""
    {main_command} 
    SELECT * 
    FROM read_csv_auto('{source_path}');
     """
    return query
    
for root,folders,files in os.walk('datalake'):
    if len(files )!=0: 
        
        for id,file in enumerate( sorted(files)):
            print('processing: ',root,file)
            #prepare query
            first_load= True if id ==0 else False
            table_name=f"raw.{os.path.basename(root)}_tripdata"
            source_path=os.path.join(root,file) 
            query=ingestion_query(source_path,table_name,first_load)
            
            #execute
            con.execute(query)


processing:  datalake\green green_tripdata_2021-01.csv
processing:  datalake\green green_tripdata_2021-02.csv
processing:  datalake\green green_tripdata_2021-03.csv
processing:  datalake\green green_tripdata_2021-04.csv
processing:  datalake\green green_tripdata_2021-05.csv
processing:  datalake\green green_tripdata_2021-06.csv
processing:  datalake\green green_tripdata_2021-07.csv
processing:  datalake\yellow yellow_tripdata_2021-01.csv
processing:  datalake\yellow yellow_tripdata_2021-02.csv
processing:  datalake\yellow yellow_tripdata_2021-03.csv
processing:  datalake\yellow yellow_tripdata_2021-04.csv
processing:  datalake\yellow yellow_tripdata_2021-05.csv
processing:  datalake\yellow yellow_tripdata_2021-06.csv
processing:  datalake\yellow yellow_tripdata_2021-07.csv


In [67]:
print ( con.execute(f"SHOW TABLES FROM raw").fetchall())

[('green_tripdata',), ('yellow_tripdata',)]


In [68]:
con.close()